# 03 — Тюнинг моделей и выбор финальной

**Цель ноутбука:** провести эксперименты с гиперпараметрами GradientBoosting,
проанализировать значимость признаков и обосновать выбор финальной модели.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

from src.data.loader import get_dataset, split_data, FEATURE_COLUMNS
from src.models.registry import build_model
from src.models.trainer import build_pipeline, cross_validate_model, evaluate_on_test

pd.set_option("display.max_columns", None)

In [ ]:
df = get_dataset(n_customers=7000, random_state=42)
X_train, X_test, y_train, y_test = split_data(df, test_size=0.2, random_state=42)

## 1. Эксперименты с гиперпараметрами GradientBoosting

Перебираем несколько конфигураций и сравниваем по ROC-AUC кросс-валидации.

In [ ]:
grid = [
    {"n_estimators": 100, "learning_rate": 0.1,  "max_depth": 3},
    {"n_estimators": 250, "learning_rate": 0.05, "max_depth": 3},
    {"n_estimators": 250, "learning_rate": 0.05, "max_depth": 4},
    {"n_estimators": 400, "learning_rate": 0.03, "max_depth": 3},
]

rows = []
for params in grid:
    params = {**params, "subsample": 0.9, "random_state": 42}
    pipe = build_pipeline(build_model("gradient_boosting", params))
    cv = cross_validate_model(pipe, X_train, y_train, n_splits=5)
    rows.append({**params, "cv_roc_auc": cv["roc_auc"], "cv_pr_auc": cv["pr_auc"]})

pd.DataFrame(rows).sort_values("cv_roc_auc", ascending=False)

**Вывод:** конфигурация `n_estimators=250, learning_rate=0.05, max_depth=3`
даёт хороший баланс качества и устойчивости — она зафиксирована в
`configs/training.yaml`. Слишком глубокие деревья склонны к переобучению,
слишком мало деревьев — недообучение.

## 2. Обучение финальной модели и значимость признаков

In [ ]:
final = build_pipeline(build_model("gradient_boosting", {
    "n_estimators": 250, "learning_rate": 0.05, "max_depth": 3,
    "subsample": 0.9, "random_state": 42,
}))
final.fit(X_train, y_train)

test_metrics = evaluate_on_test(final, X_test, y_test)
print("Финальная модель — метрики на тесте:")
for k, v in test_metrics.items():
    print(f"  {k:12s} {v:.4f}")

In [ ]:
# значимость признаков
clf = final.named_steps["classifier"]
pre = final.named_steps["preprocessor"]
names = list(pre.get_feature_names_out())
importances = clf.feature_importances_

order = np.argsort(importances)[::-1][:15]
plt.figure(figsize=(8, 6))
plt.barh([names[i] for i in order][::-1],
         [importances[i] for i in order][::-1], color="#2563eb")
plt.title("Топ-15 значимых признаков — GradientBoosting")
plt.xlabel("Значимость")
plt.tight_layout()
plt.show()

## 3. ROC-кривая финальной модели

In [ ]:
proba = final.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, proba)

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, linewidth=2, label=f"GradientBoosting (AUC={test_metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-кривая финальной модели")
plt.legend(loc="lower right")
plt.show()

## 4. Выбор финальной модели

**Финальная модель — GradientBoosting** (`n_estimators=250, learning_rate=0.05,
max_depth=3`).

Обоснование:

1. Лучший ROC-AUC и PR-AUC по кросс-валидации среди трёх кандидатов.
2. Захватывает нелинейные эффекты и взаимодействия признаков, заложенные в данные.
3. Значимость признаков согласуется с выводами EDA: главные предикторы —
   тип контракта, тип интернет-услуги, срок обслуживания, ежемесячные траты.
4. Преимущество над baseline невелико — это честно отражено в `report.md`.

Финальный пайплайн сохраняется командой `python -m src.train` в
`artifacts/model.joblib` и используется сервисом.